In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_selector as selector
from sklearn.impute import SimpleImputer

# =============================================================================
#  5 TARGET CANCER TYPES AND THEIR ALIASES
# =============================================================================
target_aliases = {
    "PRAD": [
        "PRAD",
        "prostate adenocarcinoma",
        "prostate cancer"
    ],
    "LUAD": [
        "LUAD",
        "lung adenocarcinoma",
        "lung cancer"
    ],
    "BRCA": [
        "BRCA",
        "breast invasive carcinoma",
        "breast cancer"
    ],
    "KIRC": [
        "KIRC",
        "kidney renal clear cell carcinoma",
        "kidney clear cell carcinoma"
    ],
    "COAD": [
        "COAD",
        "colon adenocarcinoma",
        "colon cancer"
    ]
}

# =============================================================================
# SELECT AND STANDARDIZE THE 5 CANCER TYPES
# =============================================================================

def select_target_cancers(dataset, label_col="Cancer_Type"):

    dataset = dataset.copy()

    if label_col not in dataset.columns:
        raise ValueError(
            f"Label column '{label_col}' not found in dataset."
        )

    # Normalize labels for matching
    normalized_labels = (
        dataset[label_col]
        .astype("string")
        .str.strip()
        .str.lower()
        .str.replace(r"[_-]+", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
    )

    alias_to_code = {}

    for code, aliases in target_aliases.items():
        for alias in aliases:
            normalized_alias = (
                alias.strip()
                .lower()
                .replace("_", " ")
                .replace("-", " ")
            )
            normalized_alias = " ".join(normalized_alias.split())
            alias_to_code[normalized_alias] = code

    # Map recognized cancer names to TCGA codes
    mapped_labels = normalized_labels.map(alias_to_code)

    # Keep only samples belonging to the 5 target cancers
    selected_mask = mapped_labels.notna()
    selected_df = dataset.loc[selected_mask].copy()

    # Standardize labels
    selected_df[label_col] = mapped_labels.loc[selected_mask].values

    print("=" * 70)
    print("5-CANCER DATASET SELECTION")
    print("=" * 70)

    print("Original dataset shape:", dataset.shape)
    print("Selected dataset shape:", selected_df.shape)
    print("Excluded samples:", int((~selected_mask).sum()))

    print("\nSelected cancer distribution:")
    print(
        selected_df[label_col]
        .value_counts()
        .reindex(target_aliases.keys(), fill_value=0)
    )

    # Check whether all target classes were found
    counts = selected_df[label_col].value_counts()
    missing_classes = [
        code for code in target_aliases
        if counts.get(code, 0) == 0
    ]

    if missing_classes:
        print("\nWARNING: These target classes were not found:")
        print(missing_classes)

        print("\nOriginal label examples:")
        print(dataset[label_col].value_counts().head(50))

        raise ValueError(
            "Some target cancer classes were not matched. "
            "Check the original Cancer_Type labels and update aliases."
        )

    return selected_df


# =============================================================================
# STRATIFIED TRAIN / VALIDATION / TEST SPLIT
# =============================================================================

def split_dataset(
    dataset,
    label_col="Cancer_Type",
    test_size=0.15,
    val_size=0.15,
    random_state=42,
):

    train_df, temp_df = train_test_split(
        dataset,
        test_size=test_size + val_size,
        stratify=dataset[label_col],
        random_state=random_state,
    )

    relative_test = test_size / (test_size + val_size)

    val_df, test_df = train_test_split(
        temp_df,
        test_size=relative_test,
        stratify=temp_df[label_col],
        random_state=random_state,
    )

    return (
        train_df.copy(),
        val_df.copy(),
        test_df.copy(),
    )


# =============================================================================
# PREPROCESSING
# =============================================================================

def preprocess(dataset):

    print("\n" + "=" * 70)
    print("SPLITTING THE 5-CANCER DATASET")
    print("=" * 70)

    train_df, val_df, test_df = split_dataset(dataset)

    print("Train:", train_df.shape)
    print("Validation:", val_df.shape)
    print("Test:", test_df.shape)

    print("\nLearning imputation statistics from training data only")

    numerical_columns = selector(
        dtype_include=["number"]
    )(train_df)

    # Do not impute categorical labels or identifiers.
    # Only numerical gene-expression features are imputed.
    excluded_columns = {"Sample_Type", "Cancer_Type"}
    numerical_columns = [
        col for col in numerical_columns
        if col not in excluded_columns
    ]

    print("Number of numerical features:", len(numerical_columns))

    gene_imputer = SimpleImputer(strategy="mean")

    train_df.loc[:, numerical_columns] = gene_imputer.fit_transform(
        train_df[numerical_columns]
    )

    val_df.loc[:, numerical_columns] = gene_imputer.transform(
        val_df[numerical_columns]
    )

    test_df.loc[:, numerical_columns] = gene_imputer.transform(
        test_df[numerical_columns]
    )

    os.makedirs("./models", exist_ok=True)

    joblib.dump(
        gene_imputer,
        "./models/gene_imputer_5_cancers.pkl"
    )

    print("\nMissing values after imputation:")
    print("Train:", train_df.isnull().sum().sum())
    print("Validation:", val_df.isnull().sum().sum())
    print("Test:", test_df.isnull().sum().sum())

    return train_df, val_df, test_df


# =============================================================================
# SAVE PROCESSED DATA
# =============================================================================

def save_processed_data(
    train_df,
    val_df,
    test_df,
    output_dir="./data/preprocessed_5_cancers",
):

    os.makedirs(output_dir, exist_ok=True)

    train_df.to_csv(
        os.path.join(output_dir, "train.csv"),
        index=False
    )

    val_df.to_csv(
        os.path.join(output_dir, "validation.csv"),
        index=False
    )

    test_df.to_csv(
        os.path.join(output_dir, "test.csv"),
        index=False
    )

    print("\n" + "=" * 70)
    print("5-CANCER DATASETS SAVED")
    print("=" * 70)
    print("Output directory:", os.path.abspath(output_dir))


# =============================================================================
# MAIN
# =============================================================================

dataset = pd.read_csv(
    "TCGA_Cancer_Dataset.csv",
    index_col=0
)

print("Original dataset shape:", dataset.shape)

# Step 1: Select only the 5 target cancer types
dataset_5 = select_target_cancers(
    dataset,
    label_col="Cancer_Type"
)

# Step 2: Split and impute
train_df, val_df, test_df = preprocess(dataset_5)

# Step 3: Inspect distributions
for split_name, split_df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"\n{split_name} shape:", split_df.shape)
    print(f"{split_name} cancer distribution:")
    print(
        split_df["Cancer_Type"]
        .value_counts()
        .reindex(target_aliases.keys(), fill_value=0)
    )

# Step 4: Save
save_processed_data(
    train_df,
    val_df,
    test_df
)

# =============================================================================
# OPTIONAL: LOAD SAVED DATA FOR GENEFORMER
# =============================================================================

output_dir = "./data/preprocessed_5_cancers"

Train_data = pd.read_csv(
    os.path.join(output_dir, "train.csv")
)

Validation_data = pd.read_csv(
    os.path.join(output_dir, "validation.csv")
)

Test_data = pd.read_csv(
    os.path.join(output_dir, "test.csv")
)

print("\nFinal saved dataset sizes:")
print("Train:", Train_data.shape)
print("Validation:", Validation_data.shape)
print("Test:", Test_data.shape)

print("\nNumber of training classes:",
      Train_data["Cancer_Type"].nunique())

assert Train_data["Cancer_Type"].nunique() == 5
assert Validation_data["Cancer_Type"].nunique() == 5
assert Test_data["Cancer_Type"].nunique() == 5

Original dataset shape: (11060, 20533)
5-CANCER DATASET SELECTION
Original dataset shape: (11060, 20533)
Selected dataset shape: (3445, 20533)
Excluded samples: 7615

Selected cancer distribution:
Cancer_Type
PRAD     550
LUAD     576
BRCA    1218
KIRC     606
COAD     495
Name: count, dtype: int64

SPLITTING THE 5-CANCER DATASET
Train: (2411, 20533)
Validation: (517, 20533)
Test: (517, 20533)

Learning imputation statistics from training data only
Number of numerical features: 20531

Missing values after imputation:
Train: 0
Validation: 0
Test: 0

Train shape: (2411, 20533)
Train cancer distribution:
Cancer_Type
PRAD    385
LUAD    403
BRCA    852
KIRC    424
COAD    347
Name: count, dtype: int64

Validation shape: (517, 20533)
Validation cancer distribution:
Cancer_Type
PRAD     83
LUAD     86
BRCA    183
KIRC     91
COAD     74
Name: count, dtype: int64

Test shape: (517, 20533)
Test cancer distribution:
Cancer_Type
PRAD     82
LUAD     87
BRCA    183
KIRC     91
COAD     74
Name: c